In [1]:
config_dir   = "/cluster/home/t144807uhn/chip-stroma-analysis/configs"
version      = "v6"
results_dir  = "/cluster/projects/kumargroup/sophia/chip-stroma-analysis/results"

In [3]:
import sys
import os
import logging

from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root / "src"))

os.chdir(Path.cwd().parent)

import wandb

import argparse as ap

from pathlib import Path

from chip_stroma.utils.header_footers import log_header, log_footer
from chip_stroma.utils.config import load_configs
from chip_stroma.utils.loggers import setup_logger
from chip_stroma.utils.io import load_overlay_arrays, load_csv_inputs
from chip_stroma.training.create_study import load_study

from chip_stroma.visualize.segmentation_plots import (
    plot_fold_boxplots,
    plot_pr_curve,
    plot_patient_dice_violin,
    plot_overlay_panel,
    plot_optuna_importance,
    plot_optuna_parallel_coords
)

logger = setup_logger(__name__)
logging.getLogger("matplotlib.category").setLevel(logging.ERROR)

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

%load_ext autoreload
%autoreload 2

In [4]:
log_header(
    pipeline_stage = "Visualization",
    config_path    = Path(config_dir) / "09_visualize.yaml",
    version        = version
)

# Load workflow and path configurations
config = load_configs(
    pipeline = Path(config_dir) / "09_visualize.yaml",
    paths    = Path(config_dir) / "00_paths.yaml"
)

# Initialize the version figure directory
evaluate_dir  = Path(config.paths.results) / version / "evaluate"
inference_dir = Path(config.paths.results) / version / "inference"
figure_dir    = Path(config.paths.figures) / version / "segmentation"
figure_dir.mkdir(parents = True, exist_ok = True) 

2026-07-30 17:54:10,152 | INFO     | utils.header_footers           | ============================================================
2026-07-30 17:54:10,153 | INFO     | utils.header_footers           | Starting Pipeline Execution
2026-07-30 17:54:10,154 | INFO     | utils.header_footers           | - Pipeline Stage    : Visualization
2026-07-30 17:54:10,154 | INFO     | utils.header_footers           | - Configurations    : chip-stroma-analysis/configs/09_visualize.yaml
2026-07-30 17:54:10,155 | INFO     | utils.header_footers           | - Version           : v6
2026-07-30 17:54:10,155 | INFO     | utils.header_footers           | - Working Directory : /cluster/home/t144807uhn
2026-07-30 17:54:10,156 | INFO     | utils.header_footers           | - Timestamp         : 2026-07-30 17:54:10
2026-07-30 17:54:10,156 | INFO     | utils.header_footers           | ============================================================
2026-07-30 17:54:10,158 | INFO     | chip_stroma.utils.config       | =

In [5]:
# 1. Per-fold macro metric distribution (excludes MACRO summary rows)
per_fold = load_csv_inputs(evaluate_dir / "per_fold_metrics.csv")

plot_fold_boxplots(
    per_fold  = per_fold[per_fold['sample_id'] != "MACRO"],
    metric    = config.visualize.fold_boxplot_metric,
    save_path = figure_dir / "fold_boxplots.png"
)

In [8]:
# 3. Precision/recall vs. threshold; justify Otsu over fixed threshold
threshold_sweep = load_csv_inputs(evaluate_dir / "threshold_sweep.csv")

plot_pr_curve(
    threshold_sweep = threshold_sweep,
    save_path       = figure_dir / "pr_curve.png"
)

In [7]:
# 4. Per-patient Dice violin plots; highlight outlier patient
per_patient = load_csv_inputs(evaluate_dir / "per_fold_metrics.csv")
per_patient = per_patient[per_patient['sample_id'] != 'MACRO']

plot_patient_dice_violin(
    per_patient       = per_patient,
    highlight_patient = config.visualize.highlight_patient,
    save_path         = figure_dir / "patient_dice_violin.png"
)

In [29]:
# 5. Overlay panels for best/median/worst QC cases selected by 08_evaluate
overlay_cases = load_csv_inputs(evaluate_dir / "overlay_cases.csv")
overlay_dir = figure_dir / "overlays"
overlay_dir.mkdir(parents = True, exist_ok = True)

for _, case in overlay_cases.iterrows():
    image, gt_mask, pred_mask = load_overlay_arrays(
        src_dir   = inference_dir,
        fold      = case['fold'],
        sample_id = case['sample_id'],
        patch_dir = config.paths.processed_data.patch_dir
    )

    plot_name = (f"{case['category']}_fold{case['fold']}_" + 
                 f"{case['sample_id']}.png")

    plot_overlay_panel(
        image, gt_mask, pred_mask,
        dice_score = case['dice'],
        save_path = overlay_dir / plot_name
    )

In [49]:
# 6. Optuna diagnostics; fANOVA importance and parallel coordinates
importance = load_csv_inputs(evaluate_dir / "optuna_importance.csv")
plot_optuna_importance(
    importance,
    save_path = figure_dir / "optuna_importance.png"
)

In [53]:
study = load_study(version, config.paths.studies)

# plot_optuna_parallel_coords(
#     study.trials_dataframe(),
#     save_path = figure_dir / "optuna_parallel_coords.png"
# )

study = study.trials_dataframe()
save_path = figure_dir / "optuna_parallel_coords.png"
